# Homework 1

In [22]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [23]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# Q1. How many lesson pages

-   24
-   72
-   240
-   720

In [24]:
print(f"{len(documents)=}")

len(documents)=72


In [25]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

# Q2. Indexing and searching
Index the documents with minsearch - make `content` a text field and `filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

-   `01-agentic-rag/lessons/03-rag.md`
-   `01-agentic-rag/lessons/14-agentic-loop.md`
-   `04-evaluation/lessons/13-llm-as-judge.md`
-   `06-best-practices/lessons/02-hybrid-search.md`

In [26]:
import requests
import minsearch

In [27]:
documents[0].get("filename")

'01-agentic-rag/lessons/01-intro.md'

In [28]:
from minsearch import Index

# 2. Create the index
index = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])


In [29]:
# 3. Fit the index
index.fit(documents)

In [30]:
# 4. Search
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query, num_results=5)

# 5. Answer Q2
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


# Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

```shell
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`), while our documents have `filename` and `content`.

Two solutions are possible:

-   Implement the RAG flow yourself
-   Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> ## Question to ask in code:> 
> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

-   700
-   7000
-   70000
-   700000

We count input tokens instead of price because the cost depends on the model and provider you use, but the size of the prompt we send is the same for everyone.

Most LLM APIs report token usage on the response object (e.g. `response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).

In [31]:
# Install if needed:
# uv add gitsource minsearch openai


import os
from dataclasses import dataclass

import minsearch
from gitsource import GithubRepositoryDataReader
from openai import OpenAI


QUESTION = "How does the agentic loop keep calling the model until it stops?"


INSTRUCTIONS = """
Your task is to answer questions from the course participants based on the provided context.
Use the context to find relevant information and provide accurate answers.
If the answer is not found in the context, respond with "I don't know."
""".strip()


PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


@dataclass
class RAGResult:
    answer: str
    usage: object


class LessonsRAG:
    def __init__(
        self,
        index: minsearch.Index,
        llm_client: OpenAI,
        model: str = "gpt-5.4-mini",
        instructions: str = INSTRUCTIONS,
        prompt_template: str = PROMPT_TEMPLATE,
    ) -> None:
        self.index = index
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.prompt_template = prompt_template

    def search(self, query: str, num_results: int = 5) -> list[dict]:
        return self.index.search(
            query=query,
            boost_dict={"content": 1.0},
            num_results=num_results,
        )

    def build_context(self, search_results: list[dict]) -> str:
        context_parts = []

        for doc in search_results:
            context_parts.append(f"filename: {doc['filename']}")
            context_parts.append(f"content: {doc['content']}")
            context_parts.append("")

        return "\n".join(context_parts).strip()

    def build_prompt(self, query: str, search_results: list[dict]) -> str:
        context = self.build_context(search_results)

        return self.prompt_template.format(
            question=query,
            context=context,
        )

    def llm(self, prompt: str):
        input_messages = [
            {
                "role": "developer",
                "content": self.instructions,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]

        # Return the whole response, not only response.output_text,
        # so we can inspect response.usage.input_tokens.
        return self.llm_client.responses.create(
            model=self.model,
            input=input_messages,
        )

    def rag(self, query: str) -> RAGResult:
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)

        return RAGResult(
            answer=response.output_text,
            usage=response.usage,
        )


def load_documents() -> list[dict]:
    reader = GithubRepositoryDataReader(
        repo_owner="DataTalksClub",
        repo_name="llm-zoomcamp",
        commit_id="8c1834d",
        allowed_extensions={"md"},
        filename_filter=lambda path: "/lessons/" in path,
    )

    files = reader.read()

    return [file.parse() for file in files]


def build_index(documents: list[dict]) -> minsearch.Index:
    index = minsearch.Index(
        text_fields=["content"],
        keyword_fields=["filename"],
    )

    index.fit(documents)

    return index


def main() -> None:
    documents = load_documents()

    print(f"Number of documents: {len(documents)}")

    index = build_index(documents)

    # Sanity check: Q2 first result
    q2_results = index.search(
        query=QUESTION,
        boost_dict={"content": 1.0},
        num_results=5,
    )

    print("Q2 first filename:", q2_results[0]["filename"])

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    rag = LessonsRAG(
        index=index,
        llm_client=client,
        model="gpt-5.4-mini",
    )

    result = rag.rag(QUESTION)

    print("\nAnswer:")
    print(result.answer)

    print("\nUsage object:")
    print(result.usage)

    print("\nInput tokens:")
    print(result.usage.input_tokens)

    print("\nClosest Q3 multiple-choice answer:")
    print("70000")


if __name__ == "__main__":
    main()

Number of documents: 72
Q2 first filename: 01-agentic-rag/lessons/14-agentic-loop.md

Answer:
It keeps calling the model in a `while True` loop. After each response, it checks whether the model returned any `function_call` items.

- If there are function calls, it runs the tools, appends the tool outputs to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the exit condition is: **no function calls this turn**.

Usage object:
ResponseUsage(input_tokens=7133, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7223)

Input tokens:
7133

Closest Q3 multiple-choice answer:
70000


# Q4. Chunking
**************************
The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding window - a window of `size` characters slides across the text in steps of `step` characters, and each window position becomes one chunk:



```python
# Q4. Chunking
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation [here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

*******************

-   Each chunk is a window of `size` characters of the page.
-   The window moves forward by `step` characters between chunks. Since `step` is smaller than `size`, consecutive chunks overlap by `size - step` (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.
-   Every chunk keeps the original fields (`filename`) and adds `start` (the offset in the page) and `content` (the chunk text).

How many chunks do you get?

-   70
-   295
-   1100
-   4500

In [32]:
from gitsource import GithubRepositoryDataReader, chunk_documents


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

chunks = chunk_documents(documents, size=2000, step=1000)

# print(len(documents))
print(f"{len(documents)=}")
# print(len(chunks))
print(f"{len(chunks)=}")

len(documents)=72
len(chunks)=295


# Q5. RAG with chunking


Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

about the same
3× fewer
10× fewer
30× fewer

In [34]:
from dataclasses import dataclass
from typing import Any

import minsearch
from gitsource import GithubRepositoryDataReader, chunk_documents
from openai import OpenAI


QUERY = "How does the agentic loop keep calling the model until it stops?"


@dataclass
class RagResult:
    answer: str
    input_tokens: int
    output_tokens: int | None
    total_tokens: int | None


class CourseRAG:
    def __init__(
        self,
        index: minsearch.Index,
        client: OpenAI,
        model: str = "gpt-5.4-mini",
        num_results: int = 5,
    ) -> None:
        self.index = index
        self.client = client
        self.model = model
        self.num_results = num_results

    def search(self, query: str) -> list[dict[str, Any]]:
        return self.index.search(
            query=query,
            num_results=self.num_results,
        )

    def build_context(self, search_results: list[dict[str, Any]]) -> str:
        context_parts = []

        for doc in search_results:
            filename = doc["filename"]
            start = doc.get("start")

            if start is None:
                header = f"filename: {filename}"
            else:
                header = f"filename: {filename}\nstart: {start}"

            context_parts.append(
                f"{header}\n\n{doc['content']}"
            )

        return "\n\n---\n\n".join(context_parts)

    def build_prompt(self, query: str, search_results: list[dict[str, Any]]) -> str:
        context = self.build_context(search_results)

        return f"""
QUESTION:
{query}

CONTEXT:
{context}
""".strip()

    def llm(self, prompt: str) -> Any:
        input_messages = [
            {
                "role": "developer",
                "content": (
                    "Your task is to answer questions from the course participants "
                    "based on the provided context. Use the context to find relevant "
                    "information and provide accurate answers. If the answer is not "
                    "found in the context, respond with \"I don't know.\""
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]

        return self.client.responses.create(
            model=self.model,
            input=input_messages,
        )

    @staticmethod
    def _get_usage_value(usage: Any, *names: str) -> int | None:
        for name in names:
            value = getattr(usage, name, None)
            if value is not None:
                return value
        return None

    def rag(self, query: str) -> RagResult:
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)

        usage = response.usage

        input_tokens = self._get_usage_value(
            usage,
            "input_tokens",
            "prompt_tokens",
        )

        output_tokens = self._get_usage_value(
            usage,
            "output_tokens",
            "completion_tokens",
        )

        total_tokens = self._get_usage_value(
            usage,
            "total_tokens",
        )

        if input_tokens is None:
            raise ValueError(f"Could not find input token usage in: {usage}")

        return RagResult(
            answer=response.output_text,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            total_tokens=total_tokens,
        )


def build_index(documents: list[dict[str, Any]]) -> minsearch.Index:
    index = minsearch.Index(
        text_fields=["content"],
        keyword_fields=["filename"],
    )
    index.fit(documents)
    return index


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    documents.append(file.parse())

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000,
)

print("Number of documents:", len(documents))
print("Number of chunks:", len(chunks))

client = OpenAI()

document_index = build_index(documents)
chunk_index = build_index(chunks)

rag_documents = CourseRAG(
    index=document_index,
    client=client,
    model="gpt-5.4-mini",
)

rag_chunks = CourseRAG(
    index=chunk_index,
    client=client,
    model="gpt-5.4-mini",
)

q3_result = rag_documents.rag(QUERY)
q5_result = rag_chunks.rag(QUERY)

print(f"\r\n{QUERY=}")

print("\r\nQ3 answer:")
print(q3_result.answer)
print("\r\Q3 input tokens:", q3_result.input_tokens)

print()
print("Q5 answer:")
print(q5_result.answer)
print("\r\nQ5 input tokens:", q5_result.input_tokens)

ratio = q3_result.input_tokens / q5_result.input_tokens

print()
print("Ratio Q3 / Q5:", ratio)
print("Chunked version sends about", round(ratio, 1), "× fewer input tokens")

<>:189: SyntaxWarning: invalid escape sequence '\Q'
<>:189: SyntaxWarning: invalid escape sequence '\Q'
/tmp/ipykernel_3367/1684673269.py:189: SyntaxWarning: invalid escape sequence '\Q'
  print("\r\Q3 input tokens:", q3_result.input_tokens)


Number of documents: 72
Number of chunks: 295

QUERY='How does the agentic loop keep calling the model until it stops?'

Q3 answer:
The loop keeps calling the model by checking whether the response contains any `function_call` items.

- It sends the current `messages` to the model.
- It runs any tool calls the model requests.
- It appends the tool outputs back into `messages`.
- It repeats in a `while True` loop.
- If a turn has no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the latest response**.
\Q3 input tokens: 7127

Q5 answer:
It keeps calling the model inside a `while True` loop. After each model response, the code checks whether there were any `function_call` items:

- if there was a function call, it runs the tool, appends the result, and loops again;
- if there were no function calls, it breaks out of the loop and stops.

So the stop condition is: **no function calls this turn**.

Q5 input tokens: 2339

Ratio Q3 / Q5: 3.04702864

# Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give the LLM a search tool and let it decide when (and what) to search. We suggest toyaikit, the small agent library from the module, but you can use anything you like - the OpenAI Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

uv add toyaikit
Create a search function that uses the chunk index. Give it a type hint and a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your search tool and run it (with toyaikit, the same way as in the ToyAIKit lesson). Use these instructions for the agent (they nudge it to search a few times):

You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

Ask it:

 - How does the agentic loop work, and how is it different from plain RAG?
 - The agent decides on its own when to search and when to answer. Count how many times it called the search tool.
 - How many times did the agent call search?

Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI gpt-5.4-mini; with a different model or provider the number may differ, so keep that in mind.

 - 0
 - 4
 - 10
 - 20


In [1]:
from __future__ import annotations

import json
from typing import Any

from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from openai import OpenAI


MODEL = "gpt-5.4-mini"

QUESTION = "How does the agentic loop work, and how is it different from plain RAG?"

INSTRUCTIONS = """
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""


def load_lesson_documents() -> list[dict[str, str]]:
    reader = GithubRepositoryDataReader(
        repo_owner="DataTalksClub",
        repo_name="llm-zoomcamp",
        commit_id="8c1834d",
        allowed_extensions={"md"},
        filename_filter=lambda path: "/lessons/" in path,
    )

    files = reader.read()

    documents = []
    for file in files:
        documents.append(file.parse())

    return documents


def build_chunk_index(
    documents: list[dict[str, str]],
    *,
    chunk_size: int = 2000,
    chunk_step: int = 1000,
) -> Index:
    chunks = chunk_documents(documents, size=chunk_size, step=chunk_step)

    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"],
    )
    index.fit(chunks)

    return index


class CourseSearch:
    def __init__(self, index: Index, num_results: int = 5) -> None:
        self.index = index
        self.num_results = num_results
        self.calls = 0

    def search(self, query: str) -> list[dict[str, Any]]:
        """
        Search the LLM Zoomcamp lesson chunks for passages relevant to the query.

        Args:
            query: Search query text.

        Returns:
            A list of matching lesson chunks with filename, chunk start offset, and content.
        """
        self.calls += 1

        results = self.index.search(
            query=query,
            boost_dict={"content": 1.0, "filename": 0.3},
            num_results=self.num_results,
        )

        return [
            {
                "filename": result["filename"],
                "start": result.get("start"),
                "content": result["content"],
            }
            for result in results
        ]


def make_search_tool_schema() -> dict[str, Any]:
    return {
        "type": "function",
        "name": "search",
        "description": (
            "Search the LLM Zoomcamp lesson chunks for passages relevant "
            "to the student's question."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text.",
                }
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    }


def run_agent(client: OpenAI, course_search: CourseSearch, question: str) -> str:
    tools = [make_search_tool_schema()]

    response = client.responses.create(
        model=MODEL,
        instructions=INSTRUCTIONS,
        input=[
            {
                "role": "user",
                "content": question,
            }
        ],
        tools=tools,
    )

    while True:
        function_calls = [
            item for item in response.output if item.type == "function_call"
        ]

        if not function_calls:
            return response.output_text

        tool_outputs = []

        for call in function_calls:
            if call.name != "search":
                raise ValueError(f"Unexpected tool call: {call.name}")

            args = json.loads(call.arguments)
            query = args["query"]

            search_result = course_search.search(query)

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(search_result),
                }
            )

        response = client.responses.create(
            model=MODEL,
            instructions=INSTRUCTIONS,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )


def main() -> None:
    load_dotenv()

    documents = load_lesson_documents()
    index = build_chunk_index(documents)

    course_search = CourseSearch(index)
    client = OpenAI()

    answer = run_agent(client, course_search, QUESTION)

    print("ANSWER")
    print(answer)
    print()
    print(f"search calls: {course_search.calls}")


if __name__ == "__main__":
    main()

ANSWER
The **agentic loop** is a repeated cycle where the LLM is in charge of deciding what to do next:

1. You send the model the user question plus the conversation history.
2. The model may respond with a **tool/function call** instead of a final answer.
3. Your code executes that tool, then sends the tool result back to the model.
4. The model can decide to call another tool, or produce the final answer.
5. Repeat until the model returns a message with **no more function calls**.

In the lesson, this is described as a simple `while True` loop:
- call the model
- run any tool calls it requested
- append results to memory
- stop when there are no function calls

So the key idea is: **the model decides how many steps are needed**.

### How it differs from plain RAG

Plain RAG is much simpler and more fixed:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

That means:

- **one r